# 🚀 ClipForge Colab 云端渲染引擎 (Cloud Engine)

**专为 Google Colab (L4/A100 GPU) 打造的商业级短视频生成后台**

- 🛡️ **双阶看门狗保护**：15分钟初始化超时熔断 + 5分钟空闲自动关机释放实例（防忘关机扣费）。
- 💾 **持久化直存 Google Drive**：成片写入 `/content/drive/MyDrive/ClipForge/outputs`，会话断开成果不丢失。
- ⚡ **Append-Only 协议通信**：抗长任务网络瞬断，本地 Web 端与 Colab 实时同步。

In [ ]:
# 1. 检测 GPU 硬件环境 (建议在 Colab 顶部「修改运行时类型」选择 NVIDIA L4)
!nvidia-smi

In [ ]:
# 2. 挂载 Google Drive 持久化目录
from google.colab import drive
import os

drive.mount('/content/drive')
output_dir = '/content/drive/MyDrive/ClipForge/outputs'
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Google Drive 挂载就绪，输出目录: {output_dir}")

In [ ]:
# 3. 准备工作区、ComfyUI与全量MiniMax/DMAD 4步模型
!mkdir -p /content/clipforge/colab
%cd /content/clipforge

# 确保通信文件存在 (Append-Only)
!touch /content/inbox.jsonl
!touch /content/outbox.jsonl

# 一键部署 ComfyUI、MiniMax-H3/Image 真实权重与 DMAD 4步极速 LoRA
!bash colab/setup_env.sh

In [ ]:
# 4. 运行引导与阶段1看门狗 (15分钟保护)
!python3 colab/bootstrap.py --session-name "clipforge_helloworld1234" --boot-timeout 15 --idle-timeout 5

In [ ]:
# 5. 启动通信监听器与视频生成服务 (阶段2空闲5分钟自动关机)
!python3 colab/runner.py --session-name "clipforge_helloworld1234" --idle-timeout 5